# Image Classification using CNN — CIFAR-10

| | |
|---|---|
| **Name** | Ayush Ranjan |
| **Roll No** | 23MIP10135 |
| **Institute** | VIT Bhopal |
| **Dataset** | [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) (also at [Kaggle](https://www.kaggle.com/c/cifar-10)) |

**Objective:** Classify 32×32 colour images into 10 object categories using a deep CNN. 
**Classes:** airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck 
**Dataset size:** 60,000 images (50,000 train + 10,000 test)


## Step 1: Install Libraries


In [ ]:
!pip install tensorflow scikit-learn matplotlib seaborn numpy -q
print('Libraries ready!')

## Step 2: Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Dense, Dropout,
                                      BatchNormalization, GlobalAveragePooling2D,
                                      Flatten)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

np.random.seed(42)
tf.random.set_seed(42)
print(f'TensorFlow {tf.__version__} | GPU: {tf.config.list_physical_devices("GPU")}')

## Step 3: Load CIFAR-10 Dataset


In [ ]:
# Load CIFAR-10 directly from TensorFlow/Keras (official source)
# Same dataset available at: https://www.kaggle.com/c/cifar-10
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']
NUM_CLASSES = len(CLASS_NAMES)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')
print(f'Pixel range: {X_train.min()} — {X_train.max()}')

## Step 4: Explore Dataset


In [ ]:
# Class distribution
train_counts = {c: int((y_train==i).sum()) for i,c in enumerate(CLASS_NAMES)}
plt.figure(figsize=(12,5))
colors = plt.cm.tab10.colors
bars = plt.bar(CLASS_NAMES, train_counts.values(), color=colors, edgecolor='k', lw=0.7)
for bar, v in zip(bars, train_counts.values()):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+30, str(v),
             ha='center', fontweight='bold', fontsize=11)
plt.title('CIFAR-10 — Training Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Class'); plt.ylabel('Images'); plt.tight_layout()
plt.savefig('cifar10_class_dist.png', dpi=150, bbox_inches='tight'); plt.show()

# Sample images (10 per class)
fig, axes = plt.subplots(10, 10, figsize=(16, 16))
for cls_idx in range(10):
    idxs = np.where(y_train.flatten()==cls_idx)[0][:10]
    for j, idx in enumerate(idxs):
        axes[cls_idx][j].imshow(X_train[idx])
        if j==0: axes[cls_idx][j].set_ylabel(CLASS_NAMES[cls_idx], fontsize=9, fontweight='bold')
        axes[cls_idx][j].axis('off')
plt.suptitle('CIFAR-10 — Sample Images (10 images × 10 classes)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('cifar10_samples.png', dpi=150, bbox_inches='tight'); plt.show()

## Step 5: Preprocessing & Augmentation


In [ ]:
IMG_SIZE   = 32   # CIFAR-10 native size (32x32)
BATCH_SIZE = 128
EPOCHS     = 100

# Normalise to [0,1]
X_train_n = X_train.astype('float32') / 255.0
X_test_n  = X_test.astype('float32')  / 255.0

# One-hot encode labels
y_train_oh = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
y_test_oh  = tf.keras.utils.to_categorical(y_test,  NUM_CLASSES)

# Augmentation for training
train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    fill_mode='reflect'
)
train_datagen.fit(X_train_n)

print(f'Train: {X_train_n.shape} | Test: {X_test_n.shape}')
print('Augmentation: rotation, shift, horizontal_flip, zoom')

## Step 6: Build Deep CNN


In [ ]:
def build_cifar_cnn(num_classes, input_shape=(32, 32, 3)):
    """
    Deep CNN for CIFAR-10 — optimised for 32x32 images
    Uses residual-style feature refinement within each block
    """
    model = Sequential([
        # Block 1 — 64 filters
        Conv2D(64,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4),input_shape=input_shape),
        BatchNormalization(),
        Conv2D(64,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        MaxPooling2D((2,2)), Dropout(0.25),

        # Block 2 — 128 filters
        Conv2D(128,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Conv2D(128,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        MaxPooling2D((2,2)), Dropout(0.3),

        # Block 3 — 256 filters
        Conv2D(256,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Conv2D(256,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Conv2D(256,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        MaxPooling2D((2,2)), Dropout(0.35),

        # Block 4 — 512 filters
        Conv2D(512,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Conv2D(512,(3,3),activation='relu',padding='same',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Dropout(0.35),

        # Classifier
        GlobalAveragePooling2D(),
        Dense(512,activation='relu',kernel_regularizer=l2(1e-4)),
        BatchNormalization(), Dropout(0.5),
        Dense(256,activation='relu',kernel_regularizer=l2(1e-4)), Dropout(0.4),
        Dense(num_classes, activation='softmax')
    ], name='CIFAR10_CNN')
    return model

model = build_cifar_cnn(NUM_CLASSES)
model.summary()
print(f'Total params: {model.count_params():,}')

## Step 7: Compile & Train


In [ ]:
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-7, verbose=1),
    ModelCheckpoint('/content/best_cifar10_cnn.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=1)
]

history = model.fit(
    train_datagen.flow(X_train_n, y_train_oh, batch_size=BATCH_SIZE),
    epochs=EPOCHS,
    validation_data=(X_test_n, y_test_oh),
    steps_per_epoch=len(X_train_n)//BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)
print(f'Best Val Accuracy: {max(history.history["val_accuracy"])*100:.2f}%')

## Step 8: Results & Evaluation


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16,6))
for ax, m, label in zip(axes, ['accuracy','loss'], ['Accuracy','Loss']):
    ax.plot(history.history[m],       'b-', label='Train', lw=2)
    ax.plot(history.history[f'val_{m}'],'r-', label='Validation', lw=2)
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel(label)
    ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('CIFAR-10 CNN — Training History', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('cifar10_history.png', dpi=150, bbox_inches='tight'); plt.show()

res = model.evaluate(X_test_n, y_test_oh, verbose=1)
print(f'Test Accuracy : {res[1]*100:.2f}%')
print(f'Top-3 Accuracy: {res[2]*100:.2f}%')
print(f'Test Loss     : {res[0]:.4f}')

In [ ]:
# Confusion matrix
y_pred = np.argmax(model.predict(X_test_n, verbose=0), axis=1)
y_true = y_test.flatten()
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(12,10))
im = ax.imshow(cm, cmap='Blues'); plt.colorbar(im)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right', fontsize=11)
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES, fontsize=11)
thresh = cm.max()/2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=9,fontweight='bold',
                color='white' if cm[i,j]>thresh else 'black')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('CIFAR-10 CNN — Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('cifar10_confusion_matrix.png', dpi=150, bbox_inches='tight'); plt.show()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Sample predictions
idxs = np.random.choice(len(X_test_n), 20, replace=False)
imgs  = X_test_n[idxs]
preds = model.predict(imgs, verbose=0)
fig, axes = plt.subplots(4, 5, figsize=(16,13)); axes = axes.flatten()
for i, idx in enumerate(idxs):
    axes[i].imshow(imgs[i])
    ti = int(y_test[idx]); pi = np.argmax(preds[i]); conf = preds[i][pi]*100
    axes[i].set_title(f'True: {CLASS_NAMES[ti]}\nPred: {CLASS_NAMES[pi]} ({conf:.1f}%)',
                      color='green' if ti==pi else 'red', fontsize=8, fontweight='bold')
    axes[i].axis('off')
plt.suptitle('CIFAR-10 — Sample Predictions', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('cifar10_predictions.png', dpi=150, bbox_inches='tight'); plt.show()

## Summary


In [ ]:
print('='*55)
print('  IMAGE CLASSIFICATION CNN (CIFAR-10) — SUMMARY')
print('='*55)
print('  Name        : Ayush Ranjan')
print('  Roll No     : 23MIP10135')
print('  Institute   : VIT Bhopal')
print('  Dataset     : CIFAR-10 (60,000 images, 10 classes)')
print(f'  Classes     : {", ".join(CLASS_NAMES)}')
print(f'  Image Size  : 32x32')
print('  Architecture: Deep CNN — 4 conv blocks + GAP')
print(f'  Parameters  : {model.count_params():,}')
print(f'  Test Accuracy: {res[1]*100:.2f}%')
print(f'  Top-3 Acc   : {res[2]*100:.2f}%')
print('='*55)